# Humanitarian Crisis Tweet Classification using HumAID
**CS439 Final Project**

We extend the classical baselines from Alam et al. (2021) by adding XGBoost, stacking ensembles, K-Means clustering, and SHAP interpretability — none of which appeared in the original HumAID paper.

**Original paper baselines (all-combined, weighted F1):**
- Random Forest: 0.707
- SVM: 0.731

**Our goal:** Beat these with XGBoost + stacking, and add interpretability via SHAP.

## 0. Imports and Setup

In [ ]:
import os
import re
import glob
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')  # non-interactive backend, saves figures to disk
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from sklearn.model_selection import RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, StackingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix
)
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
import shap

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

print('All imports successful.')

## 1. Load All Event Data

We load all TSV files from the events_set1 folder.
Each file has: tweet_id, tweet_text, class_label.
We combine all events into one dataframe.

In [ ]:
# ---- SET THIS PATH to wherever events_set1 lives on your machine ----
EVENTS_DIR = os.path.expanduser('~/Downloads/events_set1')
# ---------------------------------------------------------------------

all_files = glob.glob(os.path.join(EVENTS_DIR, '**', '*.tsv'), recursive=True)
print(f'Found {len(all_files)} TSV files across all events')

dfs = []
for f in all_files:
    try:
        df_tmp = pd.read_csv(f, sep='\t', dtype=str)
        # Only keep files that have tweet_text
        if 'tweet_text' in df_tmp.columns:
            # Extract event name and split from filename
            fname = os.path.basename(f)
            df_tmp['event'] = os.path.basename(os.path.dirname(f))
            # Determine split (train/dev/test) from filename
            if 'train' in fname:
                df_tmp['split'] = 'train'
            elif 'dev' in fname:
                df_tmp['split'] = 'dev'
            elif 'test' in fname:
                df_tmp['split'] = 'test'
            else:
                df_tmp['split'] = 'unknown'
            dfs.append(df_tmp)
    except Exception as e:
        print(f'Skipping {f}: {e}')

df = pd.concat(dfs, ignore_index=True)
print(f'\nTotal rows loaded: {len(df)}')
print(f'Columns: {df.columns.tolist()}')
df.head(3)

In [ ]:
# Drop rows missing text or label
df = df.dropna(subset=['tweet_text', 'class_label'])
df = df[df['tweet_text'].str.strip() != '']

# Remove the 'dont_know_cant_judge' class — too ambiguous, and paper removed low-prevalence classes
df = df[df['class_label'] != 'dont_know_cant_judge']

print(f'Rows after cleaning: {len(df)}')
print(f'\nClass distribution:')
print(df['class_label'].value_counts())
print(f'\nSplit distribution:')
print(df['split'].value_counts())

## 2. Text Preprocessing

Following Alam et al. (2021): remove URLs, @mentions, hashtag signs, non-ASCII characters, punctuation, and stopwords. We keep the words themselves.

In [ ]:
def clean_tweet(text):
    """Clean a tweet for classification."""
    if not isinstance(text, str):
        return ''
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    # Remove @mentions
    text = re.sub(r'@\w+', '', text)
    # Remove hashtag sign but keep the word
    text = re.sub(r'#', '', text)
    # Remove RT prefix
    text = re.sub(r'^RT\s+', '', text)
    # Remove non-ASCII characters
    text = text.encode('ascii', 'ignore').decode('ascii')
    # Remove punctuation and numbers
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    # Lowercase and strip extra whitespace
    text = text.lower().strip()
    text = re.sub(r'\s+', ' ', text)
    return text

df['clean_text'] = df['tweet_text'].apply(clean_tweet)

# Show a few examples
print('Before vs after cleaning:')
for i in range(3):
    print(f'  Original: {df["tweet_text"].iloc[i][:80]}')
    print(f'  Cleaned:  {df["clean_text"].iloc[i][:80]}')
    print()

## 3. EDA — Class Distribution and Event Coverage

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Class distribution
class_counts = df['class_label'].value_counts()
class_counts.plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Tweet Count per Humanitarian Category')
axes[0].set_xlabel('Count')
axes[0].invert_yaxis()

# Event distribution
event_counts = df['event'].value_counts()
event_counts.plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title('Tweet Count per Disaster Event')
axes[1].set_xlabel('Count')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('fig_class_event_distribution.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved fig_class_event_distribution.png')

## 4. TF-IDF Vectorization

TF-IDF converts tweet text into numerical vectors. Each dimension represents a word/n-gram, weighted by how distinctive it is across documents. We use unigrams + bigrams, matching the paper's approach.

In [ ]:
# Use official train/dev/test splits from the event files
train_df = df[df['split'] == 'train'].copy()
dev_df   = df[df['split'] == 'dev'].copy()
test_df  = df[df['split'] == 'test'].copy()

print(f'Train: {len(train_df)} | Dev: {len(dev_df)} | Test: {len(test_df)}')

# Encode labels to integers (needed for XGBoost)
le = LabelEncoder()
le.fit(df['class_label'])
print(f'\nClasses ({len(le.classes_)}): {le.classes_.tolist()}')

y_train = le.transform(train_df['class_label'])
y_dev   = le.transform(dev_df['class_label'])
y_test  = le.transform(test_df['class_label'])

# Fit TF-IDF on training set only — NEVER fit on test data (data leakage)
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),      # unigrams and bigrams
    max_features=50000,      # top 50k features
    min_df=2,                # ignore terms appearing in fewer than 2 docs
    stop_words='english',
    sublinear_tf=True        # apply log normalization to term frequencies
)

X_train = tfidf.fit_transform(train_df['clean_text'])
X_dev   = tfidf.transform(dev_df['clean_text'])
X_test  = tfidf.transform(test_df['clean_text'])

print(f'\nTF-IDF matrix shape: train={X_train.shape}, test={X_test.shape}')

## 5. K-Means Clustering

We apply unsupervised clustering to discover natural groupings in tweet language, *without* using labels. We then analyze whether these clusters align with humanitarian categories — a novel analysis not in the original paper.

In [ ]:
# Use a dense sample for clustering (sparse TF-IDF is slow for K-Means)
# We use TruncatedSVD to reduce dimensions first (faster than PCA for sparse)
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(n_components=100, random_state=RANDOM_STATE)
X_train_dense = svd.fit_transform(X_train)
print(f'Reduced to {X_train_dense.shape[1]} dimensions for clustering')

# Elbow + Silhouette
k_range = range(2, 13)
sse = []
sil_scores = []

# Sample for silhouette speed
sample_size = min(3000, len(X_train_dense))
sample_idx = np.random.choice(len(X_train_dense), sample_size, replace=False)
X_sample = X_train_dense[sample_idx]

print('Computing Elbow + Silhouette...')
for k in k_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=5, max_iter=100)
    labels = km.fit_predict(X_train_dense)
    sse.append(km.inertia_)
    sil = silhouette_score(X_sample, labels[sample_idx])
    sil_scores.append(sil)
    print(f'  k={k}: SSE={km.inertia_:.0f}, Silhouette={sil:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(list(k_range), sse, marker='o', color='steelblue')
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('SSE (Inertia)')
axes[0].set_title('Elbow Method')
axes[0].grid(True, alpha=0.3)

axes[1].plot(list(k_range), sil_scores, marker='o', color='coral')
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score by k')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('fig_elbow_silhouette.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved fig_elbow_silhouette.png')
print(f'Best silhouette at k={list(k_range)[np.argmax(sil_scores)]}')

In [ ]:
# Fit final K-Means — pick k based on the elbow/silhouette plots above
# Typically 10-11 clusters since we have 10 classes
OPTIMAL_K = 10
kmeans_final = KMeans(n_clusters=OPTIMAL_K, random_state=RANDOM_STATE, n_init=10)
cluster_labels = kmeans_final.fit_predict(X_train_dense)
train_df = train_df.copy()
train_df['cluster'] = cluster_labels

# Show cluster vs class alignment
cluster_class = pd.crosstab(train_df['cluster'], train_df['class_label'])
print('Cluster vs Class distribution (top class per cluster):')
print(cluster_class.idxmax(axis=1).to_string())

In [ ]:
# PCA visualization of clusters (2D projection)
pca = PCA(n_components=2, random_state=RANDOM_STATE)
# Use a sample for speed
viz_idx = np.random.choice(len(X_train_dense), min(5000, len(X_train_dense)), replace=False)
X_pca = pca.fit_transform(X_train_dense[viz_idx])

plt.figure(figsize=(9, 6))
for c in range(OPTIMAL_K):
    mask = cluster_labels[viz_idx] == c
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1], s=6, alpha=0.4, label=f'Cluster {c}')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)')
plt.title('Tweet Clusters — PCA 2D Visualization')
plt.legend(markerscale=2, bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.savefig('fig_pca_clusters.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved fig_pca_clusters.png')

## 6. Classification — Baseline Models

Three models: Logistic Regression (linear baseline), Random Forest (matches paper), XGBoost (gradient boosting — not in original paper).

We evaluate using **weighted F1** to match the paper's metric.

In [ ]:
def evaluate_model(model, X_test, y_test, name):
    """Train predictions and return metrics dict."""
    y_pred = model.predict(X_test)
    return {
        'name': name,
        'accuracy': accuracy_score(y_test, y_pred),
        'weighted_f1': f1_score(y_test, y_pred, average='weighted'),
        'weighted_precision': precision_score(y_test, y_pred, average='weighted'),
        'weighted_recall': recall_score(y_test, y_pred, average='weighted'),
        'y_pred': y_pred,
    }

results = {}

# Logistic Regression
print('Training Logistic Regression...')
lr = LogisticRegression(max_iter=1000, C=1.0, random_state=RANDOM_STATE, n_jobs=-1)
lr.fit(X_train, y_train)
results['Logistic Regression'] = evaluate_model(lr, X_test, y_test, 'Logistic Regression')
results['Logistic Regression']['model'] = lr
print(f'  Weighted F1: {results["Logistic Regression"]["weighted_f1"]:.4f}')

# Random Forest
print('Training Random Forest...')
rf = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train, y_train)
results['Random Forest'] = evaluate_model(rf, X_test, y_test, 'Random Forest')
results['Random Forest']['model'] = rf
print(f'  Weighted F1: {results["Random Forest"]["weighted_f1"]:.4f}')
print(f'  (Paper baseline RF: 0.707)')

# XGBoost
print('Training XGBoost...')
xgb = XGBClassifier(
    n_estimators=300,
    eval_metric='mlogloss',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    use_label_encoder=False
)
xgb.fit(X_train, y_train)
results['XGBoost'] = evaluate_model(xgb, X_test, y_test, 'XGBoost')
results['XGBoost']['model'] = xgb
print(f'  Weighted F1: {results["XGBoost"]["weighted_f1"]:.4f}')

In [ ]:
# Summary table
summary = pd.DataFrame([
    {k: v for k, v in r.items() if k not in ['model', 'y_pred', 'name']}
    for r in results.values()
], index=results.keys()).round(4)
print('Baseline results:')
print(summary)
print('\nPaper baselines for reference: RF=0.707, SVM=0.731 (weighted F1)')

In [ ]:
# Confusion matrix for best baseline model
best_baseline = max(results, key=lambda k: results[k]['weighted_f1'])
cm = confusion_matrix(y_test, results[best_baseline]['y_pred'])
cm_df = pd.DataFrame(cm, index=le.classes_, columns=le.classes_)

plt.figure(figsize=(12, 9))
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title(f'Confusion Matrix — {best_baseline}')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('fig_confusion_matrix_baseline.png', dpi=150, bbox_inches='tight')
plt.close()
print(f'Saved confusion matrix for {best_baseline}')

## 7. Hyperparameter Tuning + Stacking

In [ ]:
# Tune Random Forest
print('Tuning Random Forest...')
rf_params = {
    'n_estimators': [200, 300, 500],
    'max_depth': [None, 20, 40],
    'min_samples_split': [2, 5],
    'max_features': ['sqrt', 'log2'],
}
rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    rf_params, n_iter=8, cv=3, scoring='f1_weighted',
    random_state=RANDOM_STATE, n_jobs=-1, verbose=1
)
rf_search.fit(X_train, y_train)
rf_best = rf_search.best_estimator_
rf_tuned_f1 = f1_score(y_test, rf_best.predict(X_test), average='weighted')
print(f'Tuned RF F1: {rf_tuned_f1:.4f}')

In [ ]:
# Tune XGBoost
print('Tuning XGBoost...')
xgb_params = {
    'n_estimators': [200, 300, 500],
    'max_depth': [4, 6, 8],
    'learning_rate': [0.05, 0.1, 0.2],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
}
xgb_search = RandomizedSearchCV(
    XGBClassifier(eval_metric='mlogloss', random_state=RANDOM_STATE,
                  n_jobs=-1, use_label_encoder=False),
    xgb_params, n_iter=8, cv=3, scoring='f1_weighted',
    random_state=RANDOM_STATE, n_jobs=-1, verbose=1
)
xgb_search.fit(X_train, y_train)
xgb_best = xgb_search.best_estimator_
xgb_tuned_f1 = f1_score(y_test, xgb_best.predict(X_test), average='weighted')
print(f'Tuned XGB F1: {xgb_tuned_f1:.4f}')

In [ ]:
# Extra Trees
print('Training Extra Trees...')
et = ExtraTreesClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)
et.fit(X_train, y_train)
et_f1 = f1_score(y_test, et.predict(X_test), average='weighted')
print(f'Extra Trees F1: {et_f1:.4f}')

In [ ]:
# Stacking Classifier
print('Training Stacking Classifier...')
stack = StackingClassifier(
    estimators=[
        ('rf', rf_best),
        ('xgb', xgb_best),
        ('et', et),
    ],
    final_estimator=LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    cv=3, n_jobs=-1
)
stack.fit(X_train, y_train)
stack_pred = stack.predict(X_test)
stack_f1 = f1_score(y_test, stack_pred, average='weighted')
print(f'Stacking F1: {stack_f1:.4f}')

In [ ]:
# Final results table
final_results = pd.DataFrame([
    {'Model': 'RF (paper baseline)', 'Weighted F1': 0.707, 'Note': 'Alam et al. 2021'},
    {'Model': 'SVM (paper baseline)', 'Weighted F1': 0.731, 'Note': 'Alam et al. 2021'},
    {'Model': 'Logistic Regression (ours)', 'Weighted F1': results['Logistic Regression']['weighted_f1'], 'Note': ''},
    {'Model': 'Random Forest (ours)', 'Weighted F1': rf_tuned_f1, 'Note': 'tuned'},
    {'Model': 'XGBoost (ours)', 'Weighted F1': xgb_tuned_f1, 'Note': 'tuned'},
    {'Model': 'Extra Trees (ours)', 'Weighted F1': et_f1, 'Note': ''},
    {'Model': 'Stacking (ours)', 'Weighted F1': stack_f1, 'Note': 'RF+XGB+ET'},
]).round({'Weighted F1': 4})
print('\n=== FINAL RESULTS ===')
print(final_results.to_string(index=False))

In [ ]:
# Bar chart of all results
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['lightgray', 'lightgray', 'steelblue', 'steelblue', 'coral', 'steelblue', 'darkred']
bars = ax.barh(final_results['Model'], final_results['Weighted F1'], color=colors)
ax.axvline(x=0.731, color='gray', linestyle='--', alpha=0.7, label='SVM baseline (0.731)')
ax.set_xlabel('Weighted F1')
ax.set_title('Model Comparison — Weighted F1 on Test Set')
ax.set_xlim(0.6, 1.0)
for bar, val in zip(bars, final_results['Weighted F1']):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)
ax.legend()
plt.tight_layout()
plt.savefig('fig_model_comparison.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved fig_model_comparison.png')

## 8. SHAP Interpretability

SHAP tells us which words most strongly pushed a prediction toward each class. This is not in the original HumAID paper and is a key contribution of our work.

In [ ]:
# SHAP on XGBoost (fastest for tree-based SHAP)
# Use a small dense sample (SHAP is slow on large sparse matrices)
print('Computing SHAP values (this may take a few minutes)...')

sample_size = 500
sample_idx = np.random.choice(X_test.shape[0], sample_size, replace=False)
X_test_sample = X_test[sample_idx].toarray()  # convert sparse to dense for SHAP

explainer = shap.TreeExplainer(xgb_best)
shap_values = explainer.shap_values(X_test_sample)

# Get feature names from TF-IDF
feature_names = np.array(tfidf.get_feature_names_out())

print(f'SHAP values shape: {np.array(shap_values).shape}')
print('Done.')

In [ ]:
# Global feature importance via mean absolute SHAP values
# shap_values is list of arrays, one per class
shap_array = np.array(shap_values)  # shape: (n_classes, n_samples, n_features)
mean_abs_shap = np.abs(shap_array).mean(axis=(0, 1))  # average over classes and samples
top_idx = np.argsort(mean_abs_shap)[-20:]

plt.figure(figsize=(9, 7))
plt.barh(feature_names[top_idx], mean_abs_shap[top_idx], color='steelblue')
plt.xlabel('Mean |SHAP value|')
plt.title('Top 20 Most Influential Words (Global SHAP)')
plt.tight_layout()
plt.savefig('fig_shap_global.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved fig_shap_global.png')

In [ ]:
# Per-class top words — pick 4 interesting classes
target_classes = ['rescue_volunteering_or_donation_effort',
                  'requests_or_urgent_needs',
                  'infrastructure_and_utility_damage',
                  'injured_or_dead_people']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, cls_name in zip(axes.flat, target_classes):
    cls_idx = list(le.classes_).index(cls_name)
    class_shap = shap_array[cls_idx]  # (n_samples, n_features)
    mean_shap_cls = class_shap.mean(axis=0)
    top10 = np.argsort(np.abs(mean_shap_cls))[-10:]
    colors = ['coral' if v > 0 else 'steelblue' for v in mean_shap_cls[top10]]
    ax.barh(feature_names[top10], mean_shap_cls[top10], color=colors)
    ax.set_title(cls_name.replace('_', ' ').title(), fontsize=9)
    ax.set_xlabel('Mean SHAP value')
    ax.axvline(0, color='black', linewidth=0.5)

plt.suptitle('Per-Class SHAP: Top Predictive Words\n(coral=pushes toward class, blue=pushes away)',
             fontsize=11)
plt.tight_layout()
plt.savefig('fig_shap_per_class.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved fig_shap_per_class.png')

## 9. Summary — All Numbers for the Paper

In [ ]:
print('=' * 65)
print('PAPER NUMBERS — copy these into your writeup')
print('=' * 65)
print(f'Total tweets loaded: {len(df)}')
print(f'Train: {len(train_df)} | Dev: {len(dev_df)} | Test: {len(test_df)}')
print(f'Number of classes: {len(le.classes_)}')
print(f'TF-IDF features: {X_train.shape[1]}')
print(f'Optimal k (clusters): {OPTIMAL_K}')
print()
print('--- Baseline models ---')
for name, r in results.items():
    print(f'{name}: Weighted F1 = {r["weighted_f1"]:.4f}, Accuracy = {r["accuracy"]:.4f}')
print()
print('--- Tuned models ---')
print(f'Random Forest (tuned): {rf_tuned_f1:.4f}')
print(f'XGBoost (tuned):       {xgb_tuned_f1:.4f}')
print(f'Extra Trees:           {et_f1:.4f}')
print(f'Stacking:              {stack_f1:.4f}')
print()
print('--- Paper baselines for comparison ---')
print('Random Forest (Alam et al. 2021): 0.707')
print('SVM (Alam et al. 2021):           0.731')
print()
print('--- Figures saved ---')
saved_figs = [
    'fig_class_event_distribution.png',
    'fig_elbow_silhouette.png',
    'fig_pca_clusters.png',
    'fig_confusion_matrix_baseline.png',
    'fig_model_comparison.png',
    'fig_shap_global.png',
    'fig_shap_per_class.png',
]
for f in saved_figs:
    exists = os.path.exists(f)
    print(f'  {f}: {"OK" if exists else "MISSING"}')